<h6><center></center></h6>

<h1>
<hr style=" border:none; height:3px;">
<center>Séance n°2 : Introduction à Pandas</center>
<hr style=" border:none; height:3px;">
</h1>

**Source** : Galiana, Lino. 2023. Python Pour La Data Science. https://doi.org/10.5281/zenodo.8229676

Lors des séances précédentes, vous avez découvert la librairie Numpy. En pratique, on utilise rarement directement Numpy, qui est une librairie de bas niveau. **Pandas** est une librairie construite à partir de Numpy, qui est devenue incontournable pour la manipulation de données en Python. Elle simplifie énormément le travail de traitement et d'analyse des données.

Commençons par importer Pandas. On utilisera l'alias `pd` pour Pandas (convention usuelle).

In [ ]:
import pandas as pd
import numpy as np

## 1. Pandas & Numpy

L'objet principal en Pandas est le **DataFrame**. Il s'agit d'une structure de données à deux dimensions : lignes et colonnes. Contrairement aux arrays Numpy, **les colonnes peuvent être de types différents**.

Les DataFrames sont construits à partir d'une collection d'objets d'une dimension appelés **Series** (`pandas.Series`), qui sont des **extensions des arrays unidimensionnels Numpy**. Les Series prennent notamment en charge des **types de variables supplémentaires** par rapport à Numpy (ex: category, datetime64 et timedelta64).

Détaillons tout ça :

In [ ]:
# Création d'une Series à partir d'une liste
noms = ["Tom", "Paule", "Sophie", "Eden"]
pd.Series(noms)

*Note* : Le type `object` est un type fourre-tout pour les variables exclusivement textuelles ou mélangeant données textuelles et données numériques. Il est recommandé de convertir les variables concernées dans un type plus spécifique lorsque c'est possible (`str` ou `string` dans notre cas). Tout comme Numpy, le type peut être modifié après-coup à l'aide de la méthode `astype()`.

L'affichage précédent montre que la série créée contient à la fois des valeurs et un index *explicite* associé. La méthode `values` (resp. `index`) permet de récupérer les valeurs (resp. indices) stockées dans la Series. Ces valeurs forment un array Numpy :

In [ ]:
print(type(pd.Series(noms).values))

En revanche, que ce passe-t-il lorsqu'on affiche la même information pour l'un des types spécifiques à Pandas ?

In [ ]:
genre = ['M', 'F', 'F', 'M']
print(type(pd.Series(noms, dtype="category").values))

*Note* : Le type `category` est dédié aux variables qui prennent un nombre fini de valeurs.

Il s'agit d'un type d'array propre à Pandas, qui vient étendre les fonctionnalités de Numpy.


Au-delà de ces ajouts, la différence essentielle entre les objets Pandas et Numpy est l'**indexation**. Dans Numpy, l'indexation est implicite : on utilise des indices de position. Dans Pandas, on peut utiliser des indices de position, mais on peut aussi faire appel à des indices plus explicites : les **labels des lignes et des colonnes**. Cet accès plus intuitif aux données rend la sélection de lignes et de colonnes particulièrement aisée, comme on le verra plus tard. Le code ci-dessous permet d'illustrer le concept de label sur les lignes.

In [ ]:
# Création d'un DataFrame à partir de listes combinées

# On peut spécifier le nom des colonnes (i.e les labels des colonnes)
clients = pd.DataFrame(zip(noms, genre), columns=['nom', 'genre']) # zip permet d'associer deux listes élément par élément
print(clients)
print()

Si les indices ci-dessus semblent être de simples indices de position, ce n'est pas le cas :

In [ ]:
# On peut aussi spécifier des indices ...
## ... numériques, même désordonnés
clients = pd.DataFrame(zip(noms, genre), columns=['nom', 'genre'], index=[7,3,12,2])
print(clients)
print()

## ... ou textuels
clients = pd.DataFrame(zip(noms, genre), columns=['nom', 'genre'], index=['client_7','client_3','client_12','client_2'])
print(clients)
print()

Il est possible de modifier les indices *après* la création du DataFrame :

In [ ]:
clients = pd.DataFrame(zip(noms, genre), columns=['nom', 'genre'])
print(clients)
print()

# On peut modifier les indices après la création du DataFrame

## En utilisant une ou plusieurs colonnes du DataFrame
print("Avec set_index() :")
clients = clients.set_index("nom")
print(clients)
print()

## En réinitialisant les indices aux indices par défaut
print("Avec reset_index() :")
clients = clients.reset_index()
print(clients)

Cette façon de procéder pour créer un DataFrame est purement illustrative. En général, le DataFrame est créé à partir de données préexistantes, telles que des données stockées dans un fichier. C'est ce qu'on va voir dans la section 2.

## 2. Importer des données

Pandas permet de lire des données stockées sous différents formats. Il prend en charge le format **`CSV` et ses dérivés (`TXT`, `TSV` ...)**, les données issues de tableurs **Excel** ou LibreOffice, le format **`Parquet`** et les formats de type **`JSON`**. Nous nous concentrerons sur la lecture de fichiers `CSV` et dérivés.

Pour créer un DataFrame à partir d'un fichier stocké **en local** ou **disponible sur internet**, utilisez la commande **`pd.read_csv()`**. Cette fonction possèdent de nombreux paramètres. Les plus importants sont probablement les suivants :    
- **`sep`** : Délimiteur du fichier. Par exemple : `","` pour un fichier `CSV` (valeur par défaut) et `"\t"` pour un fichier `TXT`.
- `header` : Numéro de la ligne contenant le nom des colonnes et marquant le début des données. Par défaut, Pandas déduira le nom des colonnes à partir de la première ligne du fichier (i.e `header=0`). Lorsqu'un fichier ne spécifie pas le nom des colonnes, passez `header=None` et utilisez le paramètre `names` pour les définir.
- `names` : Définir le nom des colonnes lorsqu'elles sont absentes ou lorsqu'on souhaite remplacer celles du fichier (dans ce cas, passez `header=0`).  
- **`dtype`** : Indiquer le type d'une ou plusieurs colonnes du DataFrame  
- Autres paramètres utiles : `usecols`, `skiprows`, `skip_blank_lines` (`True` par défaut), `quoting`.

Si vous ne connaissez pas le délimiteur du fichier, utilisez la fonction `open()` pour lire les premières lignes du fichier avant d'utiliser `pd.read_csv()`. Par exemple :

In [ ]:
# Lecture des deux premières lignes de `temperature.csv`
nlines = 2
with open('sample_data/temperature.csv', 'r') as file: # 'r' pour 'read' (lire)
  for i in range(nlines):
    line = next(file).strip() # strip() supprime tous les espaces en début et fin de chaîne
    print(line)

Vous pouvez désormais créer un DataFrame à partir du fichier `temperature.csv` dont le délimiteur est un point-virgule. Vous pouvez indiquer à Pandas que les colonnes `Ville` et `Région` contiennent des chaînes de caractères.

In [ ]:
temperature = pd.read_csv('sample_data/temperature.csv', sep=';', dtype={'Ville':'string','Région':'string'})
temperature.head(3)

Pandas permet également de collecter des données en ligne non directement disponibles pour le téléchargement (**web scrapping**). La commande **`pd.read_html()`** permet par exemple de récupérer les tableaux HTML présents sur une page web. Elle les renvoie sous la forme d'une liste de DataFrame.

Les données de `temperature.csv` nous fournissent des informations sur la température moyenne au cours d'une journée pour chaque mois. Imaginons que nous souhaitions plutôt connaître la température moyenne la plus haute et la température moyenne la plus basse par mois dans plusieurs villes d'Europe. Le site [Current Results](https://www.currentresults.com/Weather/Europe/Cities/average-european-city-weather.php) fournit ces informations sous forme de tables. Nous pouvons récupérer les tables de températures pour le mois de Janvier ainsi :

In [ ]:
#url = r'https://www.currentresults.com/Weather/Europe/Cities/temperature-january.php'
#tables = pd.read_html(url)
#print(f"{len(tables)} tables ont été récupérées sur la page")
#print()
#print(f"La première table est la suivante :")
#print(tables[0])

Cet outil a ses limites. Il peut par exemple échouer lorsqu'un site web n'autorise pas la connexion sans identification (user agent). Dans ce cas, il faudra passer par d'autres outils comme `requests` ou `BeautifulSoup`.

In [ ]:
# Wikipédia n'autorise pas l'accès sans identification

#url = r'https://en.wikipedia.org/wiki/List_of_cities_by_average_temperature'
#tables = pd.read_html(url)

## 3. Propriétés d'un DataFrame

Pour afficher les premières lignes d'un DataFrame, utilisez la méthode **`<DataFrame>.head()`** en spécifiant le nombre de lignes que vous souhaitez afficher. De la même façon, vous pouvez afficher les dernières lignes d'un DataFrame à l'aide de **`<DataFrame>.tail()`**. Remplacez `<DataFrame>` par le nom de votre DataFrame.

Voici quelques fonctions utiles pour obtenir des informations sur un DataFrame :

In [ ]:
# Même fonctions qu'en Numpy :
print("* Nombre de dimensions :", temperature.ndim)
print("* Nombre d'éléments :", temperature.size)
print("* Nombre de lignes et de colonnes :", temperature.shape)
print("* Nombre de lignes :", len(temperature)) # ou temperature.shape[0]
# Fonction propre à Pandas
print("* Nombre de colonnes :", len(temperature.columns)) # ou temperature.shape[1]

In [ ]:
print("* Type des éléments :")
print(temperature.dtypes)

In [ ]:
print("* Liste des colonnes :")
print(temperature.columns)
print()

print("* Liste des indices :")
print(temperature.index)

**Exercice 1** :
1. Créez un DataFrame à partir du fichier `pulse.txt`.
2. Combien de lignes et de colonnes contient-il ?
3. Affichez le nom des colonnes. Pouvez-vous deviner quel genre d'information ce jeu de données contient ?
4. Affichez le type des colonnes. Selon vous, en vous fiant au nom des colonnes, le type de chacune des colonnes est-il le type le plus adapté aux données qu'elles contiennent ?   

Lors de la préparation des données, il est important d'examiner les types des objets Pandas que vous manipulez et de les convertir si besoin, en fonction de l'usage que vous comptez faire de ces données.

In [ ]:
# Question 1


# Question 2


# Question 3


# Question 4



## 4. Indexation et slicing d'un DataFrame

Pour **accéder à un ou plusieurs éléments** d'un DataFrame, il existe deux manières de procéder :
- `<DataFrame>.`**`iloc`**`[<lignes>, <colonnes>]` : cette méthode utilise les **indices de position**
- `<DataFrame>.`**`loc`**`[<lignes>, colonnes>]` : cette méthode utilise les **labels** des lignes et des colonnes

Pour bien comprendre, nous allons utiliser le DataFrame suivant :

In [ ]:
# Céation d'un DataFrame à partir d'une liste de listes
df = pd.DataFrame([[1,12,2003,21],[4,5,1996,29],[7,8,1967,58],[10,11,2012,12]],
                  columns=["day","month","year","age"],
                  index=["Tom", "Paule", "Sophie", "Eden"])
print(df)

*Rappel* :
- Le slicing permet de récupérer un sous ensemble d'éléments. La syntaxe générale est la suivante : `[<début>:<fin>:<pas>]`. Si `début` n'est pas spécifié, c'est l'indice minimal qui sera utilisé. Si `fin` n'est pas spécifié, c'est l'indice maximal qui sera utilisé. Si vous ne mentionnez pas le `pas`, c'est un pas de 1 qui sera appliqué.
- Le symbole "`:`" utilisé seul permet d'extraire tous les éléments de la dimension, c'est-à-dire tous les éléments d'une ligne ou d'une colonne

### 4.1 Accéder à des colonnes

In [ ]:
print(df)

Pour accéder à une colonne, on peut utiliser l'**indice** (de position) de la colonne. La colonne `month` est la deuxième colonne du DataFrame, elle a donc pour indice 1 :

In [ ]:
df.iloc[:,1]

L'objet retourné est une Série :

In [ ]:
print(type(df.iloc[:,1]))

Pour accéder à une colonne, on peut aussi utiliser le **nom** de la colonne. Dans ce cas, **il n'est plus nécessaire de connaitre la position de la colonne** dans le DataFrame, ce qui **facilite** les opérations :

In [ ]:
df.loc[:,"month"]

De même que pour les array Numpy, on peut sélectionner **plusieurs colonnes simultanément**. Soit via des **opération de slicing**, soit en spécifiant une **liste de valeurs**. Par exemple :

In [ ]:
# De la deuxième (indice 1) à la quatrième colonne (indice 3)
# Attention ! Quatrième colonne non comprise
print(df.iloc[:,1:3]) # ou df.iloc[:,[1,2]]

L'objet retourné est un DataFrame :

In [ ]:
print(type(df.iloc[:,1:3]))

In [ ]:
# Du mois ("month") à l'âge ("age")
# Attention ! Âge compris
print(df.loc[:,"month":"age"])

In [ ]:
# Première (indice 0) et troisième (indice 2) colonnes
print(df.iloc[:,[0,2]])

In [ ]:
# Jour ("day") et année ("year")
print(df.loc[:,["day","year"]])

**ATTENTION !**  

Les exemples ci-dessus mettent en évidence une différence fondamentale entre `iloc` et `loc` lors des opérations de slicing :
- Pour `iloc`, le ou les éléments associés à l'**indice de fin** du slicing sont **exclus** des résultats
- Pour `loc`, le ou les éléments associés au **label de fin** du slicing sont **inclus** dans les résultats

Enfin, il est aussi possible d'accéder aux colonnes par leur label ainsi :

In [ ]:
df["age"]

In [ ]:
df.age

La dernière méthode requiert néanmoins d'avoir des noms de colonnes sans espace ou caractères spéciaux.

### 4.2 Accéder à des lignes

In [ ]:
print(df)

Pour accéder à une ligne, on peut utiliser l'**indice** de la ligne. `Paule` correspond à la deuxième ligne du DataFrame, c'est-à-dire à l'indice 1 :

In [ ]:
df.iloc[1,:]

L'objet retourné est une Série :

In [ ]:
print(type(df.iloc[1,:]))

Pour accéder à une ligne, on peut aussi utiliser le **label** de la ligne. Dans ce cas, **il n'est plus nécessaire de connaitre la position de la ligne** dans le DataFrame. Cette méthode est **plus fiable** car la **position** des lignes **peut évoluer au cours du traitement**, après des opérations de tri, de suppression etc.

In [ ]:
df.loc["Paule",:]

On peut sélectionner **plusieurs lignes simultanément**. Soit via des **opération de slicing**, soit en spécifiant une **liste de valeurs**. Par exemple :

In [ ]:
# De la deuxième (indice 1) à la quatrième ligne (indice 3)
# Attention ! Quatrième ligne non comprise
print(df.iloc[1:3,:])

In [ ]:
# De "Paule" à "Eden"
# Attention ! "Eden" compris
print(df.loc["Paule":"Eden",:])

In [ ]:
# Première (indice 0) et troisième (indice 2) lignes
print(df.iloc[[0,2],:])

In [ ]:
# "Tom" et "Sophie"
print(df.loc[["Tom","Sophie"],:])

### 4.3 Accéder à des éléments

Il est bien sûr possible de faire des sélections de **colonnes et** de **lignes** dans une **même instruction**.

Nous pouvons par exemple récupérer l'année de naissance et l'âge de "Eden" et "Paule" :

In [ ]:
print(df.loc[["Eden","Paule"],["year","age"]])

Ou les deux premières lignes des deux premières colonnes :

In [ ]:
print(df.iloc[:2,:2])

### 4.4 Autre exemple

Le DataFrame utilisé jusqu'ici a été choisi de façon à faciliter la compréhension des différents concepts : il marque clairement la distinction entre les indices de position et les labels pour les lignes. La plupart du temps, cependant, **les labels des lignes sont aussi numériques**. Il faut alors faire attention à **ne pas confondre les méthodes `iloc` et `loc`**, car aucune erreur ne sera levée.

Dans l'exemple précédent, si vous tapez `df.loc[0,:]` une erreur `KeyError` est levée.

À l'inverse, voyez ce qu'il se passe quand les labels sont numériques :

In [ ]:
# Création du dataset
df = pd.DataFrame([[3,4],[7,8]], columns=["a","b"], index=[1,3])
print(df)

In [ ]:
# Avec iloc et l'indice 1 (i.e deuxième ligne)
print(df.iloc[1,:])

In [ ]:
# Avec loc et l'indice 1 (i.e première ligne)
print(df.loc[1,:])

Aucune erreur n'est renvoyée, puisque l'indice 1 existe dans les deux cas. Par contre, les résultats sont différents. Attention aux mauvaises surprises !

### 4.5 Filtre logique

Enfin, il est possible de sélectionner des éléments à partir de **conditions logiques**, soit en spécifiant ces conditions **entre crochets**, sans utiliser ni `iloc` ni `loc`, soit en utilisant **`loc`**. Utiliser `loc` permet, en plus de la sélection sur les lignes, de réaliser une sélection sur les colonnes, ou inversement.

In [ ]:
df = pd.DataFrame([[1,12,2003,21],[4,5,1996,29],[7,8,1967,58],[10,11,2012,12]],
                  columns=["day","month","year","age"],
                  index=["Tom", "Paule", "Sophie", "Eden"])
print(df)

In [ ]:
# Noms de toutes les personnes de plus de 25 ans

print("Personnes de plus de 25 ans :")
print(df[df["age"] > 25])
print()

# `index` permet d'accéder aux indices, qui contiennent les prénoms
# `tolist()` permet de transformer l'index en liste

print("Nom des personnes de plus de 25 ans :")
print(df[df["age"] > 25].index.tolist())

In [ ]:
# Année de naissance de toutes les personnes de 21 ans ou moins

print("Année de naissance des personnes de moins de 21 ans :")
print(df.loc[df["age"] <= 21, "year"])
print()

# Pour ne récupérer que les années de naissance, utilisez `values`
# `tolist()` permet de transformer l'array Numpy en liste

print("Avec `values` :")
print(df.loc[df["age"] <= 21, "year"].values.tolist())

In [ ]:
# Noms de toutes les personnes nées en décembre

print("Nom des personnes nées en décembre :")
print(df[df["month"] == 12].index.tolist())

Vous pouvez créer des filtres logiques plus complexes en utilisant des **opérateurs** :
* **`&`** pour **AND**
* **`|`** pour **OR**
* **`~`** pour **NOT**   

Dans ce cas, afin d'éviter les mauvaises surprises, il est primordial d'utiliser des parenthèses pour délimiter les différentes conditions.

In [ ]:
# Age des personnes nées entre 1950 et 2000

print("Age des personnes nées entre 1950 et 2000 :")
print(df.loc[(df["year"] >= 1950) & (df["year"] <= 2000), "age"].values.tolist())

Un certain nombre de fonctions logiques sont déjà implémentées en Pandas. Lorsque c'est possible, il est recommandé de les utiliser plutôt que de les coder soi-même car elles sont optimisées. Par exemple : `isna()` (ou `isnull()`), `any()`, `all()`, `isin()`.

In [ ]:
# Y a-t-il des personnes de moins de 20 ans dans le jeu de données ?

print((df["age"] <= 20).any()) # ou any(df["age"] <= 20)

Notez que les valeurs manquantes en Pandas sont affichées `NaN` ou `<NA>` (et `NaT` pour les variables temporelles, c'est-à-dire de type `datetime64`). Vous pouvez utiliser la commande `pd.NA` pour les créer.

In [ ]:
# Valeurs manquantes dans un DataFrame
print(pd.DataFrame([[1,pd.NA,3.1],[3,5,pd.NA]], columns=["a","b","c"]))

**Exercice 2** :
À l'aide du jeu de données `temperature.csv`, répondez aux questions suivantes :
1. Affichez les 5 premières lignes
2. Quelle ville se trouve à la vingt-deuxième ligne ?
3. Modifiez le DataFrame pour que la colonne `Ville` devienne l'index du DataFrame
4. Quelle température fait-il en moyenne à Budapest en Décembre ?
5. Entre Madrid et Rome, quelle est la ville qui a la température moyenne à l'année la plus élevée ?
6. Quelle température fait-il en mars, en juillet, et en novembre à Moscou ?
7. Listez toutes les villes d'Europe du Nord. Combien y en a-t-il dans le jeu de données ?
8. Quelles sont les villes situées entre les latitudes 45.996014 et 52.147855, et les longitudes -0.523470 et 11.547644 ?
9. Quelles villes ne sont ni à l'Ouest, ni au Sud de l'Europe ? Proposez deux façons de procéder.
10. La température moyenne en Avril est-elle positive pour l'ensemble des villes ? Proposez deux façons de procéder.

In [ ]:


# Question 1


# Question 2


# Question 3


# Question 4


# Question 5


# Question 6


# Question 7


# Question 8


# Question 9


# Question 10



**ATTENTION !**   

Il est très difficile de prévoir quand les opérations de filtrage effectuées sur un DataFrame renverront **une vue ou une copie** de l'objet initial. Pour rappel : toute modification affectuée sur une *vue* modifiera le DataFrame original. Dans le doute, je vous conseille de **systématiquement utiliser `<DataFrame>.copy()`** lorsque vous souhaitez **réaliser des modifications** sur le DataFrame obtenu à l'issue d'un filtrage **sans modifer le DataFrame original**. Bien sûr, si a contrario votre but est de modifier le DataFrame original, n'utilisez pas `copy()`. À titre purement illustratif, car même cette méthode n'est pas toujours exacte, observez ce que retourne l'attribut `_is_view` :

In [ ]:
df = pd.DataFrame([[1,2],[3,4]], columns=["a","b"])
print("slicing sans copie, est-ce une vue ? ", df.iloc[:,1:3]._is_view)
print("slicing avec copie, est-ce une vue ? ", df.iloc[:,1:3].copy()._is_view)

# 5. Modification d'un DataFrame

On peut ajouter, supprimer, modifier et renommer des lignes et des colonnes entières. On peut également modifier quelques éléments indépendamment.

## 5.1 Colonnes

Pour **ajouter ou modifier une colonne**, il suffit de spécifier le nom de la colonne entre crochets, avec ou sans `loc`, et de lui assigner des valeurs. On peut créer des colonnes à partir de colonnes déjà existantes. On peut appliquer des fonctions à des colonnes entières.

In [ ]:
df = pd.DataFrame([[1,2,3],[2,2,3],[3,2,3]], columns=["a","b","c"])
print(df)

In [ ]:
# Créer ou modifier une colonne à l'aide d'une valeur unique
df["d"] = 1
print(df)

In [ ]:
# Créer ou modifier une colonne en spécifiant une valeur pour chaque ligne
df["d"] = [4,5,6]
print(df)

In [ ]:
# Créer ou modifier une colonne à partir des valeurs contenues dans d'autres colonnes
df["d"] = df["a"] * df["b"]
print(df)

In [ ]:
# Céer une colonne en ne spécifiant de valeurs que pous certaines lignes
df.loc[1, "e"] = 7
print("Création :")
print(df)
print()

In [ ]:
# Créer ou modifier une colonne à partir de colonnes déjà existantes et d'une fonction
df["d"] = np.exp(df["a"] - df["c"])
print(df)

In [ ]:
# Modifier plusieurs colonnes
df[["b","d"]] = np.log(df[["b","d"]])
print(df)

In [ ]:
df[["b","d"]] = [[1,2],[1,2],[1,2]]
print(df)

Il est aussi possible d'**ajouter des colonnes** en concaténant plusieurs DataFrames, ou des DataFrames et des Series, à l'aide de la fonction **`pd.concat()`**, en passant **1** au paramètre **`axis`**. La **concaténation** utilise les labels des lignes.

In [ ]:
df1 = pd.DataFrame([[1,2,3],[1,2,3],[1,2,3]], columns=["a","b","c"])
df2 = pd.DataFrame([[4,5],[4,5],[4,5]], columns=["d","e"])
print("Premier DataFrame ou Series :")
print(df1)
print("Second DataFrame ou Series :")
print(df2)
print()

print("Concaténation column-wise :")
df = pd.concat([df1,df2], axis=1)
print(df)

Lorsque les labels des lignes ne correspondent pas, des valeurs manquantes sont ajoutées par Pandas :

In [ ]:
df1 = pd.DataFrame([[1,2,3],[1,2,3],[1,2,3]], columns=["a","b","c"])
df2 = pd.DataFrame([[4,5],[4,5],[4,5]], columns=["d","e"], index=[0,3,4])
print("Premier DataFrame ou Series :")
print(df1)
print("Second DataFrame ou Series :")
print(df2)
print()

print("Concaténation column-wise :")
df = pd.concat([df1,df2], axis=1)
print(df)

Pour **supprimer des colonnes**, on peut utiliser `<DataFrame>.drop()`, ou remplacer le DataFrame par ce même DataFrame dont on ne garde qu'un sous-ensemble de colonnes.

In [ ]:
df = pd.DataFrame([[1,2,3,4],[2,2,3,4],[3,2,3,4]], columns=["a","b","c","d"])
print("Avant modification :")
print(df)
print()

# Supprimer des colonne avec drop()
df = df.drop(columns=["a","d"]) # voir aussi le paramètre `inplace`
print("Après modification :")
print(df)

In [ ]:
df[["a","d"]] = 2
print("Avant modification :")
print(df)
print()

# Supprimer des colonnes en spécifiant le sous-ensemble de colonnes à garder
df = df[["b","c"]]
print("Après modification :")
print(df)

Il est aussi possible de supprimer des colonnes à l'aide de filtres logiques.

In [ ]:
df = pd.DataFrame([[1,4,5],[2,pd.NA,4],[1,3,6]], columns=["a","b","c"])
print("Avant modification :")
print(df)
print()

# Supprimer les colonnes contenant des valeurs manquantes
df = df.dropna(axis=1)
print("Après modification :")
print(df)
print()

# On aurait aussi pu taper : df.loc[:,~df.isna().any(axis=0)]

On peut **renommer des colonnes** avec `<DataFrame>.rename()`.

In [ ]:
print("Avant modification :")
print(df)
print()
df = df.rename(columns={"c":"e"}) # voir aussi le paramètre `inplace`
print("Après modification :")
print(df)

## 5.2 Lignes

Les opérations d'ajout, de suppression, de modification et de renommage sont similaires à ce qui a été vu pour les colonnes.

In [ ]:
df = pd.DataFrame([[1,1,1],[2,2,2],[3,3,3]], columns=["a","b","c"], index=[0,4,5])
print(df)

Pour **ajouter une ligne**, on peut utiliser `loc`, spécifier le label de la nouvelle ligne et lui assigner des valeurs. On peut créer des lignes à partir de lignes déjà existantes. On peut appliquer des fonctions à des lignes entières.

In [ ]:
# Ajouter une ligne
df.loc[6,:] = [4,5,6]
df = df.astype('int')
print(df)

On ne peut pas ajouter plusieurs lignes à la fois avec cette méthode. Pour **ajouter plusieurs lignes** simultanément, il faut concaténer plusieurs DataFrames, ou des DataFrames et des Series, à l'aide de la fonction **`pd.concat()`**, en passant **0** au paramètre **`axis`**. La concaténation utilise le nom des colonnes. Lorsque les colonnes ne correspondent pas, des valeurs manquantes sont ajoutées. Si `ignore_index=True`, les labels originaux des lignes ne sont pas conservés, les lignes sont labellisées de 0 à n-1.

In [ ]:
df2 = pd.DataFrame([[0,0,0],[-1,-1,-1]], columns=["a","b","c"])
print("Premier DataFrame ou Series :")
print(df)
print("Second DataFrame ou Series :")
print(df2)
print()

print("Concaténation row-wise :")
df = pd.concat([df,df2], axis=0)
print(df)

Pour **modifier une ligne**, on peut utiliser `loc` ou `iloc`. On peut aussi appliquer des fonctions à des lignes entières.

In [ ]:
# Modifier une ligne
df.iloc[2,:] = 4
df.loc[4,:] = [5,6,7]
print(df)

In [ ]:
# Modifier plusieurs lignes
df.loc[[5,6],:] = df.loc[[5,6],:]**2
print(df)

Pour **supprimer** des lignes on peut utiliser `<DataFrame>.drop()`, ou remplacer le DataFrame par ce même DataFrame dont on ne garde qu'un sous-ensemble de lignes.

In [ ]:
# Supprimer des lignes avec drop
df = df.drop(index=[0,6])
print(df)

In [ ]:
# Supprimer des lignes en spécifiant le sous-ensemble de lignes à conserver
df = df.loc[[1,5],:]
print(df)

Il est aussi possible de supprimer des lignes à l'aide de filtres logiques.

In [ ]:
df = pd.DataFrame([[1,4,5],[2,pd.NA,4],[1,3,6]], columns=["a","b","c"])
print("Avant modification :")
print(df)
print()

# Supprimer les lignes contenant des 1 dans la colonne "a"
df = df[~(df["a"] == 1)]
print("Après modification :")
print(df)
print()

In [ ]:
df = pd.DataFrame([[1,4,5],[2,pd.NA,4],[1,3,6]], columns=["a","b","c"])
print("Avant modification :")
print(df)
print()

# Supprimer les lignes contenant des valeurs manquantes
df = df.dropna(axis=0)
print("Après modification :")
print(df)
print()

# On aurait aussi pu taper : df[~df.isna().any(axis=1)]

On peut **renommer des lignes** avec `<DataFrame>.reset_index()` (vu précédemment) ou `<DataFrame>.rename()`.

In [ ]:
print("Avant modification :")
print(df)
print()
df = df.rename(index={2:1}) # voir aussi le paramètre `inplace`
print("Après modification :")
print(df)

## 5.3 Éléments

On peut modifier certains éléments du DataFrame en utilisant l'indexation et l'affectation. On peut modifier une seule valeur ou plusieurs valeurs simultanément. Ci-dessous, quelques exemples de ce qu'il est possible de faire.

In [ ]:
df = pd.DataFrame([[1,12,2003,21,-355],[4,5,1996,29,1090],[7,8,1967,58,12397],[10,11,2012,12,25]],
                  columns=["day","month","year","age","solde"],
                  index=["Tom", "Paule", "Sophie", "Eden"])
print("Avant modification :")
print(df)
print()

# Modifier une valeur
df.iloc[0,1] = 6 # ou df.loc["Tom","month"]
df.loc["Paule","day"] *= 2 # ou df.loc["Paule","day"] * 2

# Modifier plusieurs valeurs
df.loc[["Tom","Paule"],["year","age"]] += 1
df.loc["Eden", ["day","month"]] = [24,6]
df.loc[["Tom","Eden"],"solde"] = df.loc[["Tom","Eden"],"solde"].abs() # ou np.abs(df.loc["Tom","solde"])

print("Après modification :")
print(df)
print()

In [ ]:
# Modifier plusieurs valeurs à l'aide de filtres logiques
df.loc[df["month"] == 6, "month"] = 12
print(df)

## 5.6 Autre

Pandas propose quelques méthodes qui permettent de réorganiser les données au sein d'un DataFrame, comme :
- `sort_values()` : trier les données selon les valeurs contenues dans une ou plusieurs colonnes (ou une ou plusieurs lignes)
- `sort_index()` : trier les données selon les labels des lignes (ou des colonnes)
- `T` (ou `transpose()`) : transposer les lignes et les colonnes

In [ ]:
df = pd.DataFrame([[1,5,5],[2,4,5],[3,3,6]], columns=["a","b","c"], index=[2,0,1])
print("Avant modification :")
print(df)
print()

# Trier les données selon les valeurs contenues dans une ou plusieurs colonnes
df = df.sort_values(by=["b"])
print("Tri selon les valeurs de `b` :")
print(df)
print()

# Trier les données selon le label des lignes
df = df.sort_index(axis=0)
print("Tri selon les labels des lignes :")
print(df)
print()

# Statistiques simples sur un DataFrame

Pour les statistiques descriptives classiques, Pandas propose un certain nombre de méthodes, qui peuvent être appliquées à un DataFrame et / ou une Series. Voici une liste non exhaustive :
- `unique()` :  lister les valeurs uniques
- `count()` : compter le nombre d'éléments qui ne sont pas des valeurs manquantes pour chaque ligne ou colonne
- `nunique()` : compter le nombre d'éléments distincts pour chaque ligne ou colonne  
- `value_counts()` : compter le nombre d'éléments par valeur unique
- `sum()` : sommer par ligne ou par colonne  
- `mean()` : calculer la moyenne par ligne ou par colonne
- `min()` : trouver la valeur minimale par ligne ou par colonne
- `argmin()` (Series) : trouver l'indice de la valeur minimale de la Series
- `max()` : trouver la valeur maximale par ligne ou par colonne
- `argmax()` (Series) : trouver l'indice de la valeur maximale de la Series
- `std()` : calculer l'écart-type par ligne ou par colonne
- `median()` : calculer la médiane par ligne ou par colonne
- `corr()` : calculer la corrélation deux à deux entre plusieurs colonnes

La méthode `agg()` permet de calculer une ou plusieurs statistiques pour une ou plusieurs colonnes à la fois.  

In [ ]:
df = pd.DataFrame([[1,12,2003,21,-355],[4,5,1996,29,1090],[7,8,1967,58,12397],[10,11,2012,12,25]],
                  columns=["day","month","year","age","solde"],
                  index=["Tom", "Paule", "Sophie", "Eden"])
print(df)

In [ ]:
# Age moyen
print(f"L'âge moyen est de {df['age'].mean()} ans")

# Année et âge minimums
print()
print("Âge et année minimums :")
print(df[["year","age"]].min())

In [ ]:
# Age minimal et maximal
# Solde minimal, maximal et somme
stats = df.agg({
                "age" : ["min", "max"],
                "solde" : ["min", "max", "sum"]
               }
              )
print(stats)

**Exercice 3** :
Répondez aux questions suivantes à l'aide du jeu de données `pulse.txt`. 

`pulse1` stocke les pouls au repos. `pulse2` stocke les pouls au repos **ou** après avoir couru selon le groupe auquel chaque étudiant a été assigné.

1. Supprimez la colonne `Year` qui n'apporte aucune information utile
2. Y a-t-il des valeurs manquantes dans certaines colonnes ? Si oui combien ?
3. À la question précédente, vous avez pu constater que certaines lignes présentent des valeurs manquantes. Ces lignes n'étant pas nombreuses, supprimez-les du jeu de données.
4. Quels sont les âges représentés dans les données et combien y a-t-il d'observations par âge ? Que remarquez-vous ?
5. Pour garantir la fiabilité de l'analyse, écartez les valeurs aberrantes en ne conservant que les individus de 26 ans ou moins.
6. Calculez la taille moyenne des individus associés à `gender = 1`
7. Calculez la taille moyenne des individus associés à `gender = 2`
8. En vous basant sur les deux questions précédentes, devinez quel genre représente chacun des chiffres. Pour plus de lisibilité, changer le type de la colonne `gender` pour `string` et remplacez les nombres 1 et 2 par "F" et "M" en fonction de vos conclusions.
9. Affichez les informations des 5 étudiants les plus petits.
10. À partir des colonnes `ran` et `pulse2`, déterminez quel chiffre dans `ran`, 1 ou 2, correspond au groupe qui a couru. Remplacez les chiffres par "yes" ou "no" en fonction de vos conclusions.
11. L'Indice de Masse Corporelle (IMC) se calcule en divisant le poids en kilos par le carré de la taille **en mètres**. Créez une colonne `bmi` (body mass index).
12. Existe-t-il une corrélation entre le pouls au repos et l'IMC ? Puis entre le pouls **après la course** et l'IMC ?
13. Un IMC en dessous de 18.5 est considéré comme insuffisant (`under`), entre 18.5 et 24.9 comme normal (`normal`), entre 25 et 30 comme en surpoids (`over`), et au-delà de 30 comme obésité (`obese`). Créez une nouvelle colonne `bmi_cat` avec ces catégories. Combien y a-t-il d'étudiants dans chaque catégorie ?
14. Existe-t-il une différence de pouls entre les hommes et les femmes ? Utilisez la médiane.
15. Afficher la taille et le poids moyen par genre. Vous pouvez utiliser la méthode `groupby()`
16. Les femmes boivent-elles plus que les hommes dans cet échantillon ?

# Bonus

In [ ]:
url = "https://koumoul.com/s/data-fair/api/v1/datasets/igt-pouvoir-de-rechauffement-global/convert"
emissions = pd.read_csv(url)
emissions.head(2)

1. Les deux premiers chiffres du code INSEE correspondent au département. Créez une nouvelle colonne contenant les départements pour chaque ligne. Utilisez la méthode `str`
2. Quel est le département avec le plus de communes ?